# Segundo Parcial Big Data - Cloud Provider Analytics

MVP tecnico end-to-end: `Landing -> Bronze -> Silver -> Gold -> Serving Cassandra/AstraDB`.

Este notebook esta preparado para Google Colab y usa PySpark DataFrames, Parquet, Structured Streaming y CQL.

## 0. Setup

In [16]:
# En Colab, descomentar si hace falta instalar dependencias.
# %pip install -q pyspark==3.5.1 cassandra-driver==3.29.2

In [17]:
import os
import sys
import importlib
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if 'src.tp2_pipeline' in sys.modules:
    importlib.reload(sys.modules['src.tp2_pipeline'])

def configure_java_for_spark_notebook():
    if os.environ.get('JAVA_HOME'):
        return os.environ['JAVA_HOME']
    candidates = [
        Path('/Users/juan/Library/Java/JavaVirtualMachines/corretto-21.0.2/Contents/Home'),
        Path('/Users/juan/Library/Java/JavaVirtualMachines/corretto-1.8.0_352/Contents/Home'),
    ]
    for java_home in candidates:
        if (java_home / 'bin' / 'java').exists():
            os.environ['JAVA_HOME'] = str(java_home)
            os.environ['PATH'] = f"{java_home / 'bin'}:{os.environ.get('PATH', '')}"
            return str(java_home)
    return None

selected_java = configure_java_for_spark_notebook()
print('Repo root:', REPO_ROOT)
print('JAVA_HOME para Spark:', selected_java)

Repo root: /Users/juan/Documents/GitHub/big-data-tp2
JAVA_HOME para Spark: /Users/juan/Library/Java/JavaVirtualMachines/corretto-21.0.2/Contents/Home


## 1. Paths y configuracion

In [18]:
DATASET_ROOT = REPO_ROOT / 'cloud_provider_challenge_dataset_v1'
LANDING = DATASET_ROOT / 'datalake' / 'landing'
DATALAKE_OUT = DATASET_ROOT / 'datalake'
CHECKPOINT_OUT = DATASET_ROOT / 'checkpoints'

print('Landing existe:', LANDING.exists(), LANDING)
print('Archivos landing:')
for p in sorted(LANDING.glob('*')) if LANDING.exists() else []:
    print(' -', p.relative_to(DATASET_ROOT))

Landing existe: True /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/landing
Archivos landing:
 - datalake/landing/billing_monthly.csv
 - datalake/landing/customers_orgs.csv
 - datalake/landing/marketing_touches.csv
 - datalake/landing/nps_surveys.csv
 - datalake/landing/resources.csv
 - datalake/landing/support_tickets.csv
 - datalake/landing/usage_events_stream
 - datalake/landing/users.csv


Si el dataset no existe en Colab, subir el zip provisto y descomprimirlo antes de continuar. La estructura esperada es `cloud_provider_challenge_dataset_v1/datalake/landing/`.

## 2. SparkSession

In [19]:
from src.tp2_pipeline import Paths, build_spark

paths = Paths(LANDING, DATALAKE_OUT, CHECKPOINT_OUT)
spark = build_spark('big-data-tp2-notebook')
spark

Spark 3.5.1 inicializado con Java 21.0.2


## 3. Schemas explicitos

In [20]:
from src.tp2_pipeline import CSV_SPECS, USAGE_SCHEMA

print('CSV schemas:')
for name, (schema, keys) in CSV_SPECS.items():
    print('\n', name, 'dedupe keys=', list(keys))
    print(schema.simpleString())

print('\nUsage stream schema:')
print(USAGE_SCHEMA.simpleString())

CSV schemas:

 customers_orgs dedupe keys= ['org_id']
struct<org_id:string,org_name:string,industry:string,hq_region:string,plan_tier:string,is_enterprise:string,signup_date:string,sales_rep:string,lifecycle_stage:string,marketing_source:string,nps_score:string>

 users dedupe keys= ['user_id']
struct<user_id:string,org_id:string,email:string,role:string,active:string,created_at:string,last_login:string>

 resources dedupe keys= ['resource_id']
struct<resource_id:string,org_id:string,service:string,region:string,created_at:string,state:string,tags_json:string>

 billing_monthly dedupe keys= ['invoice_id']
struct<invoice_id:string,org_id:string,month:string,subtotal:string,credits:string,taxes:string,currency:string,exchange_rate_to_usd:string>

 support_tickets dedupe keys= ['ticket_id']
struct<ticket_id:string,org_id:string,category:string,severity:string,created_at:string,resolved_at:string,csat:string,sla_breached:string>

 nps_surveys dedupe keys= ['org_id', 'survey_date']
struct<o

## 4. Batch -> Bronze

In [21]:
from src.tp2_pipeline import ingest_batch_to_bronze

ingest_batch_to_bronze(spark, paths)

for table in ['customers_orgs', 'users', 'billing_monthly', 'resources']:
    p = paths.bronze / table
    if p.exists():
        df = spark.read.parquet(str(p))
        print(f'bronze/{table}:', df.count(), 'rows')
        df.printSchema()

bronze_batch.customers_orgs: 80 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/customers_orgs


bronze_batch.users: 800 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/users


bronze_batch.resources: 400 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/resources
bronze_batch.billing_monthly: 240 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/billing_monthly
bronze_batch.support_tickets: 1000 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/support_tickets
bronze_batch.nps_surveys: 92 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/nps_surveys
bronze_batch.marketing_touches: 1500 rows -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/marketing_touches
bronze/customers_orgs: 80 rows
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise:

## 5. Streaming -> Bronze

In [22]:
from src.tp2_pipeline import ingest_usage_stream_to_bronze

# Usa trigger availableNow=True para que Colab procese los JSONL disponibles y termine.
ingest_usage_stream_to_bronze(spark, paths)

bronze_stream = spark.read.parquet(str(paths.bronze / 'usage_events_stream'))
print('bronze/usage_events_stream rows:', bronze_stream.count())
bronze_stream.printSchema()
bronze_stream.select('event_id', 'event_ts', 'usage_date', 'schema_version', 'value', 'cost_usd_increment').show(10, truncate=False)

26/06/11 19:48:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


bronze_stream.usage_events -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/usage_events_stream


bronze/usage_events_stream rows: 8415
root
 |-- event_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: string (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: string (nullable = true)
 |-- genai_tokens: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- source_file: string (nullable = false)
 |-- usage_date: date (nullable = true)

+----------------+-------------------+----------+--------------+-------+------------------+
|event_id        |event_ts           |usage_date|schema_version|value  |cost_usd_increment|
+----------------+-------------------+----------+--------------+----

## 6. Silver + calidad + quarantine

In [23]:
from src.tp2_pipeline import build_silver_events

build_silver_events(spark, paths)

silver_events = spark.read.parquet(str(paths.silver / 'usage_events'))
quarantine_events = spark.read.parquet(str(paths.quarantine / 'usage_events'))

print('silver/usage_events rows:', silver_events.count())
silver_events.printSchema()

print('quarantine/usage_events rows:', quarantine_events.count())
quarantine_events.select('event_id', 'usage_date', 'schema_version', 'value', 'unit', 'cost_usd_increment', 'dq_reason').show(20, truncate=False)

silver.usage_events: 7998 rows


quarantine.usage_events: 417 rows


silver/usage_events rows: 7998
root
 |-- org_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: double (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- value_num: double (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- state: string (nullable = true)
 |-- requests: 

## 7. Gold mart FinOps

In [24]:
from src.tp2_pipeline import build_gold_finops

gold_mart = build_gold_finops(spark, paths)
gold_mart = spark.read.parquet(str(paths.gold / 'org_daily_usage_by_service'))

print('gold/org_daily_usage_by_service rows:', gold_mart.count())
gold_mart.printSchema()
gold_mart.orderBy('org_id', 'usage_date', 'service').show(20, truncate=False)

gold.org_daily_usage_by_service: 5424 rows
gold/org_daily_usage_by_service rows: 5424
root
 |-- org_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- event_count: long (nullable = true)
 |-- requests: double (nullable = true)
 |-- total_value: double (nullable = true)
 |-- cpu_hours: double (nullable = true)
 |-- storage_gb_hours: double (nullable = true)
 |-- genai_tokens: double (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- daily_cost_usd: double (nullable = true)
 |-- anomaly_event_count: long (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- usage_date: date (nullable = true)

+------------+---------+-------------+---------+---------+-----------+--------+------------------+---------+------------------+------------+---------+--------------+-------------------+--------------------------+----------+
|org_id

## 8. Idempotencia

In [25]:
def count_path(path):
    return spark.read.parquet(str(path)).count() if path.exists() else 0

before = {
    'bronze_usage_events_stream': count_path(paths.bronze / 'usage_events_stream'),
    'silver_usage_events': count_path(paths.silver / 'usage_events'),
    'gold_org_daily_usage_by_service': count_path(paths.gold / 'org_daily_usage_by_service'),
}

# Reprocesamos Silver y Gold. Ambas capas usan overwrite, por lo que no duplican.
build_silver_events(spark, paths)
build_gold_finops(spark, paths)

after = {
    'bronze_usage_events_stream': count_path(paths.bronze / 'usage_events_stream'),
    'silver_usage_events': count_path(paths.silver / 'usage_events'),
    'gold_org_daily_usage_by_service': count_path(paths.gold / 'org_daily_usage_by_service'),
}

print('Conteos antes:', before)
print('Conteos despues:', after)
print('Idempotencia OK:', before == after)

# Gold fue reescrito en modo overwrite; recargamos el DataFrame para evitar referencias a archivos viejos.
spark.catalog.clearCache()
gold_mart = spark.read.parquet(str(paths.gold / 'org_daily_usage_by_service'))

silver.usage_events: 7998 rows


quarantine.usage_events: 417 rows


gold.org_daily_usage_by_service: 5424 rows


Conteos antes: {'bronze_usage_events_stream': 8415, 'silver_usage_events': 7998, 'gold_org_daily_usage_by_service': 5424}
Conteos despues: {'bronze_usage_events_stream': 8415, 'silver_usage_events': 7998, 'gold_org_daily_usage_by_service': 5424}
Idempotencia OK: True


## 9. Cassandra / AstraDB

In [26]:
# Carga .env del repo y detecta secure-connect-*.zip en la raiz (local).
# En Colab: subi el zip y setea las variables a mano o en Secrets.
from src.astra_config import configure_astra_env

astra_env = configure_astra_env(REPO_ROOT)
ASTRA_CLIENT_ID = astra_env['client_id']
ASTRA_CLIENT_SECRET = astra_env['client_secret']
ASTRA_SECURE_CONNECT_BUNDLE = astra_env['secure_connect_bundle']
ASTRA_KEYSPACE = astra_env['keyspace']

print('ASTRA_KEYSPACE:', ASTRA_KEYSPACE)
print('Bundle:', ASTRA_SECURE_CONNECT_BUNDLE)
print('Bundle configurado:', bool(ASTRA_SECURE_CONNECT_BUNDLE))
print('Credenciales configuradas:', bool(ASTRA_CLIENT_ID and ASTRA_CLIENT_SECRET))
if not (ASTRA_CLIENT_ID and ASTRA_CLIENT_SECRET):
    print('Falta completar ASTRA_CLIENT_ID y ASTRA_CLIENT_SECRET en .env (copiar desde .env.example)')

ASTRA_KEYSPACE: default_keyspace
Bundle: /Users/juan/Documents/GitHub/big-data-tp2/secure-connect-tp2-bigdata.zip
Bundle configurado: True
Credenciales configuradas: True


In [27]:
# Recargamos Gold desde Parquet porque pasos previos pueden haber sobrescrito la carpeta.
spark.catalog.clearCache()
gold_mart = spark.read.parquet(str(paths.gold / 'org_daily_usage_by_service'))
gold_mart.orderBy('org_id', 'usage_date', 'service').show(20, truncate=False)


+------------+---------+-------------+---------+---------+-----------+--------+------------------+---------+------------------+------------+---------+--------------+-------------------+--------------------------+----------+
|org_id      |service  |org_name     |plan_tier|hq_region|event_count|requests|total_value       |cpu_hours|storage_gb_hours  |genai_tokens|carbon_kg|daily_cost_usd|anomaly_event_count|updated_at                |usage_date|
+------------+---------+-------------+---------+---------+-----------+--------+------------------+---------+------------------+------------+---------+--------------+-------------------+--------------------------+----------+
|org_0lvsnujz|analytics|Nova Cloud 45|standard |us-east  |1          |0.0     |12.7808           |0.0      |12.7808           |0.0         |0.0      |1.0161        |0                  |2026-06-11 22:50:55.293193|2025-07-03|
|org_0lvsnujz|database |Nova Cloud 45|standard |us-east  |1          |0.0     |1.7515            |1.7515

In [28]:
def connect_astra():
    from cassandra.auth import PlainTextAuthProvider
    from cassandra.cluster import Cluster
    from src.astra_schema import ensure_astra_schema

    if not (ASTRA_CLIENT_ID and ASTRA_CLIENT_SECRET and ASTRA_SECURE_CONNECT_BUNDLE):
        raise RuntimeError('Faltan ASTRA_CLIENT_ID, ASTRA_CLIENT_SECRET o ASTRA_SECURE_CONNECT_BUNDLE')

    cloud_config = {'secure_connect_bundle': ASTRA_SECURE_CONNECT_BUNDLE}
    auth_provider = PlainTextAuthProvider(ASTRA_CLIENT_ID, ASTRA_CLIENT_SECRET)
    cluster = Cluster(
        cloud=cloud_config,
        auth_provider=auth_provider,
        connect_timeout=30,
        control_connection_timeout=30,
    )
    session, resolved_keyspace = ensure_astra_schema(
        cluster,
        ASTRA_KEYSPACE,
        ASTRA_SECURE_CONNECT_BUNDLE,
        REPO_ROOT,
    )
    session.default_timeout = 60
    print('Conectado a keyspace:', resolved_keyspace)
    return cluster, session

def load_daily_mart_to_cassandra(session, df, batch_size=500, concurrency=8):
    from cassandra.concurrent import execute_concurrent_with_args

    cql = '''
    INSERT INTO org_daily_usage_by_service (
      org_id, usage_date, service, org_name, plan_tier, hq_region,
      event_count, requests, total_value, cpu_hours, storage_gb_hours,
      genai_tokens, carbon_kg, daily_cost_usd, anomaly_event_count, updated_at
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    '''
    prepared = session.prepare(cql)
    columns = [
        'org_id', 'usage_date', 'service', 'org_name', 'plan_tier', 'hq_region',
        'event_count', 'requests', 'total_value', 'cpu_hours', 'storage_gb_hours',
        'genai_tokens', 'carbon_kg', 'daily_cost_usd', 'anomaly_event_count', 'updated_at',
    ]

    def row_args(row):
        return tuple(getattr(row, col) for col in columns)

    def flush(batch):
        if not batch:
            return 0
        results = execute_concurrent_with_args(
            session,
            prepared,
            batch,
            concurrency=concurrency,
            raise_on_first_error=False,
        )
        failures = [result for success, result in results if not success]
        if failures:
            raise RuntimeError(f'Cassandra load failed for {len(failures)} rows. First error: {failures[0]}')
        return len(batch)

    written = 0
    batch = []
    for row in df.select(*columns).toLocalIterator():
        batch.append(row_args(row))
        if len(batch) >= batch_size:
            written += flush(batch)
            batch = []
    written += flush(batch)
    print(f'Carga Cassandra: {written} filas escritas en org_daily_usage_by_service')

print('Funcion de carga Cassandra lista.')

Funcion de carga Cassandra lista.


In [29]:
cluster, session = connect_astra()
load_daily_mart_to_cassandra(session, gold_mart)
session.shutdown()
cluster.shutdown()
print('Carga Cassandra finalizada')

Keyspace 'default_keyspace' ya existe.
Tabla lista: org_daily_usage_by_service
Conectado a keyspace: default_keyspace
Carga Cassandra: 5424 filas escritas en org_daily_usage_by_service
Carga Cassandra finalizada


## 10. Consultas CQL

In [30]:
for cql_file in ['cql/01_create_keyspace.cql', 'cql/02_create_tables.cql', 'cql/03_queries_demo.cql']:
    cql_path = REPO_ROOT / cql_file
    print('\n' + '=' * 80)
    print(cql_path)
    print('=' * 80)
    print(cql_path.read_text())


/Users/juan/Documents/GitHub/big-data-tp2/cql/01_create_keyspace.cql
CREATE KEYSPACE IF NOT EXISTS cloud_analytics
WITH replication = {
  'class': 'NetworkTopologyStrategy',
  'datacenter1': 3
};


/Users/juan/Documents/GitHub/big-data-tp2/cql/02_create_tables.cql
USE cloud_analytics;

CREATE TABLE IF NOT EXISTS org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    org_name text,
    plan_tier text,
    hq_region text,
    event_count bigint,
    requests double,
    total_value double,
    cpu_hours double,
    storage_gb_hours double,
    genai_tokens double,
    carbon_kg double,
    daily_cost_usd double,
    anomaly_event_count bigint,
    updated_at timestamp,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC);


/Users/juan/Documents/GitHub/big-data-tp2/cql/03_queries_demo.cql
USE cloud_analytics;

-- Consulta 1:
-- Costos y requests diarios por org y servicio en rango de fechas.
SELECT
  

In [31]:
cluster, session = connect_astra()
org_id = gold_mart.select('org_id').first()['org_id']
min_date, max_date = gold_mart.agg({'usage_date': 'min'}).first()[0], gold_mart.agg({'usage_date': 'max'}).first()[0]
rows_1 = session.execute('''
  SELECT org_id, usage_date, service, requests, daily_cost_usd, event_count
  FROM org_daily_usage_by_service
  WHERE org_id = %s AND usage_date >= %s AND usage_date <= %s
''', (org_id, min_date, max_date))
print('Consulta 1 (muestra):')
print(list(rows_1)[:20])
sample_date = gold_mart.filter(gold_mart.org_id == org_id).select('usage_date').first()['usage_date']
rows_2 = session.execute('''
  SELECT org_id, usage_date, service, requests, genai_tokens, carbon_kg, daily_cost_usd, anomaly_event_count, event_count
  FROM org_daily_usage_by_service
  WHERE org_id = %s AND usage_date = %s
''', (org_id, sample_date))
print('Consulta 2 (servicios y anomalias):')
print(list(rows_2))
session.shutdown()
cluster.shutdown()

Keyspace 'default_keyspace' ya existe.
Tabla lista: org_daily_usage_by_service
Conectado a keyspace: default_keyspace
Consulta 1 (muestra):
[Row(org_id='org_l58kxo9t', usage_date=Date(20331), service='compute', requests=247.0, daily_cost_usd=19.7902, event_count=5), Row(org_id='org_l58kxo9t', usage_date=Date(20331), service='storage', requests=109.0, daily_cost_usd=4.553, event_count=3), Row(org_id='org_l58kxo9t', usage_date=Date(20330), service='compute', requests=359.0, daily_cost_usd=28.6123, event_count=6), Row(org_id='org_l58kxo9t', usage_date=Date(20330), service='storage', requests=236.0, daily_cost_usd=5.3785, event_count=3), Row(org_id='org_l58kxo9t', usage_date=Date(20329), service='compute', requests=108.0, daily_cost_usd=8.3301, event_count=2), Row(org_id='org_l58kxo9t', usage_date=Date(20328), service='storage', requests=0.0, daily_cost_usd=0.2038, event_count=1), Row(org_id='org_l58kxo9t', usage_date=Date(20327), service='compute', requests=0.0, daily_cost_usd=1.3837, eve

## 11. Evidencias finales

In [32]:
evidence_paths = {
    'bronze_customers_orgs': paths.bronze / 'customers_orgs',
    'bronze_usage_events_stream': paths.bronze / 'usage_events_stream',
    'silver_usage_events': paths.silver / 'usage_events',
    'quarantine_usage_events': paths.quarantine / 'usage_events',
    'gold_org_daily_usage_by_service': paths.gold / 'org_daily_usage_by_service',
}

for name, path in evidence_paths.items():
    print('\n', name, '->', path)
    if path.exists():
        df = spark.read.parquet(str(path))
        print('rows:', df.count())
        df.show(5, truncate=False)
    else:
        print('NO EXISTE')


 bronze_customers_orgs -> /Users/juan/Documents/GitHub/big-data-tp2/cloud_provider_challenge_dataset_v1/datalake/bronze/customers_orgs
rows: 80
+------------+--------------+----------+----------+----------+-------------+-----------+---------+---------------+----------------+---------+--------------------------+------------------------------------------------------------------------------------------------------------------------+-----------+
|org_id      |org_name      |industry  |hq_region |plan_tier |is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|ingest_ts                 |source_file                                                                                                             |ingest_date|
+------------+--------------+----------+----------+----------+-------------+-----------+---------+---------------+----------------+---------+--------------------------+-----------------------------------------------------------------------------------

rows: 8415
+----------------+--------------------+------------+------------+---------+------------+----------------+------+--------+------------------+--------------+---------+------------+-------------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|event_id        |timestamp           |org_id      |resource_id |service  |region      |metric          |value |unit    |cost_usd_increment|schema_version|carbon_kg|genai_tokens|event_ts           |ingest_ts             |source_file                                                                                                                                     |usage_date|
+----------------+--------------------+------------+------------+---------+------------+----------------+------+--------+------------------+--------------+---------+------------+-------------------+----------------------+------------

rows: 7998
+------------+----------------+--------------------+------------+----------+------------+----------------+-------+--------+------------------+--------------+---------+------------+-------------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------------+---------+--------------+--------+------------+----------+---------------+----------+--------+---------+----------------+--------------+-----------+---------------+----------+
|org_id      |event_id        |timestamp           |resource_id |service   |region      |metric          |value  |unit    |cost_usd_increment|schema_version|carbon_kg|genai_tokens|event_ts           |ingest_ts              |source_file                                                                                                                                     |value_num|org_name      |industry|hq_region   |plan_tier |lifecycle_stage|s